# Gymnasium single-symbol environment design
Milestone 5A validates the production `single_symbol_env_v1` simulator. At step *t*, the agent sees data through date *t*, trades at date *t+1* open, and is valued at date *t+1* close. This notebook does not train PPO.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)
print('Python executable:', sys.executable)

Project root: /Users/m.abdulbasit/Downloads/virtual-trader
Python executable: /Users/m.abdulbasit/Downloads/virtual-trader/.venv/bin/python


## Load one configurable eligible symbol dataset

In [2]:
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from data_pipeline.src.config import AI_MINIMUM_USABLE_ROWS, PROCESSED_SYMBOLS_DIR
from reinforcement_learning.environments import SingleSymbolEnvConfig, SingleSymbolTradingEnv
from reinforcement_learning.environments.config import DEFAULT_OBSERVATION_FEATURES
from reinforcement_learning.environments.validation import validate_environment
from reinforcement_learning.evaluation import BuyAndHoldPolicy, RandomPolicy, run_baseline

preferred_symbol = 'MCB'
paths = sorted(PROCESSED_SYMBOLS_DIR.glob('*.csv'))
eligible = []
for path in paths:
    candidate = pd.read_csv(path, dtype={'symbol': 'string'})
    if len(candidate) >= AI_MINIMUM_USABLE_ROWS:
        eligible.append((path, candidate))
selected = next(((p, d) for p, d in eligible if p.stem == preferred_symbol), eligible[0] if eligible else None)
demonstration_mode = selected is None
if demonstration_mode:
    display(Markdown('### ⚠️ Demonstration mode — no local symbol meets the configured readiness threshold. This deterministic fixture validates mechanics only and is **not suitable for research conclusions**.'))
    rows = 80
    dates = pd.bdate_range('2025-01-01', periods=rows)
    close = 100 + np.linspace(0, 12, rows) + np.sin(np.arange(rows) / 4)
    dataset = pd.DataFrame({'symbol': 'DEMO', 'date': dates, 'open': close - 0.3, 'high': close + 1, 'low': close - 1, 'close': close, 'volume': 100_000 + np.arange(rows) * 100})
    for position, column in enumerate(DEFAULT_OBSERVATION_FEATURES): dataset[column] = np.sin(np.arange(rows) / (position + 2))
else:
    dataset_path, dataset = selected
    display(Markdown(f'### Production dataset: {dataset_path.stem}'))
print('Rows:', len(dataset), 'Range:', dataset['date'].min(), 'to', dataset['date'].max())
print('Feature columns:', list(DEFAULT_OBSERVATION_FEATURES))

### ⚠️ Demonstration mode — no local symbol meets the configured readiness threshold. This deterministic fixture validates mechanics only and is **not suitable for research conclusions**.

Rows: 80 Range: 2025-01-01 00:00:00 to 2025-04-22 00:00:00
Feature columns: ['simple_return', 'log_return', 'high_low_range', 'open_close_return', 'rolling_volatility_20', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'atr_14', 'obv', 'volume_ma_20']


## Configure and instantiate the production environment

In [3]:
config = SingleSymbolEnvConfig()
env = SingleSymbolTradingEnv(dataset, config)
print('Version:', config.environment_version)
print('Action space:', env.action_space)
print('Observation space:', env.observation_space)
print('Observation features:', env.observation_feature_names)

Version: single_symbol_env_v1
Action space: Discrete(3)
Observation space: Box(-3.4028235e+38, 3.4028235e+38, (17,), float32)
Observation features: ('simple_return', 'log_return', 'high_low_range', 'open_close_return', 'rolling_volatility_20', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'atr_14', 'obv', 'volume_ma_20', 'portfolio_cash_ratio', 'portfolio_position_value_ratio', 'portfolio_position_indicator', 'portfolio_unrealized_return_ratio', 'portfolio_current_drawdown')


## Reset and inspect one observation

In [4]:
observation, reset_info = env.reset(seed=42)
display(pd.Series(observation, index=env.observation_feature_names, name='Value'))
display(reset_info)

simple_return                        0.0
log_return                           0.0
high_low_range                       0.0
open_close_return                    0.0
rolling_volatility_20                0.0
rsi_14                               0.0
macd                                 0.0
macd_signal                          0.0
macd_histogram                       0.0
atr_14                               0.0
obv                                  0.0
volume_ma_20                         0.0
portfolio_cash_ratio                 1.0
portfolio_position_value_ratio       0.0
portfolio_position_indicator         0.0
portfolio_unrealized_return_ratio    0.0
portfolio_current_drawdown           0.0
Name: Value, dtype: float32

{'environment_version': 'single_symbol_env_v1',
 'date': Timestamp('2025-01-01 00:00:00'),
 'next_date': Timestamp('2025-01-02 00:00:00'),
 'action': None,
 'action_name': None,
 'execution_price': None,
 'shares_traded': 0,
 'transaction_cost': 0.0,
 'cash': 1000000.0,
 'shares_held': 0,
 'portfolio_value': 1000000.0,
 'realized_profit_loss': 0.0,
 'unrealized_profit_loss': 0.0,
 'drawdown': 0.0,
 'reward_components': {'portfolio_growth': 0.0,
  'transaction_cost_penalty': 0.0,
  'drawdown_penalty': 0.0,
  'invalid_action_penalty': 0.0}}

## Manual Buy, Hold, Sell transitions

In [5]:
env.reset(seed=42)
manual_info = []
for action in (1, 0, 2):
    _, reward, terminated, truncated, info = env.step(action)
    manual_info.append(info)
display(manual_info)
display(env.get_history())

[{'environment_version': 'single_symbol_env_v1',
  'date': Timestamp('2025-01-01 00:00:00'),
  'next_date': Timestamp('2025-01-02 00:00:00'),
  'action': 1,
  'action_name': 'Buy',
  'execution_price': 100.14935234477845,
  'shares_traded': 9975,
  'transaction_cost': 1498.2350618226096,
  'cash': 11.22057119582314,
  'shares_held': 9975,
  'portfolio_value': 1001494.2649381774,
  'realized_profit_loss': 0.0,
  'unrealized_profit_loss': 1494.2649381774245,
  'drawdown': 0.0,
  'reward_components': {'portfolio_growth': 0.0014931496352252624,
   'transaction_cost_penalty': -0.0,
   'drawdown_penalty': -0.0,
   'invalid_action_penalty': -0.0}},
 {'environment_version': 'single_symbol_env_v1',
  'date': Timestamp('2025-01-02 00:00:00'),
  'next_date': Timestamp('2025-01-03 00:00:00'),
  'action': 0,
  'action_name': 'Hold',
  'execution_price': None,
  'shares_traded': 0,
  'transaction_cost': 0.0,
  'cash': 11.22057119582314,
  'shares_held': 9975,
  'portfolio_value': 1005323.8700656082,

,initial_portfolio_value,observation_date,execution_date,action,action_name,execution_price,shares_traded,transaction_cost,cash,shares_held,portfolio_value,realized_profit_loss,unrealized_profit_loss,drawdown,reward
0,1000000.0,2025-01-01,2025-01-02,1,Buy,100.149352,9975,1498.235062,1.122057e+01,9975,1.001494e+06,0.000000,1494.264938,0.000000,0.001493
1,1000000.0,2025-01-02,2025-01-03,0,Hold,NaN,0,0.000000,1.122057e+01,9975,1.005324e+06,0.000000,5323.870066,0.000000,0.003817
2,1000000.0,2025-01-03,2025-01-06,2,Sell,100.786916,-9975,1508.275698,1.004355e+06,0,1.004355e+06,4355.361125,0.000000,0.000963,-0.001060


## Complete deterministic baselines

In [6]:
buy_hold = run_baseline(SingleSymbolTradingEnv(dataset, config), BuyAndHoldPolicy(), seed=42)
random_run = run_baseline(SingleSymbolTradingEnv(dataset, config), RandomPolicy(seed=42), seed=42)
comparison = pd.DataFrame([
    {'Baseline': 'Buy and Hold', **{k: v for k, v in buy_hold.metrics.items() if k != 'daily_returns'}},
    {'Baseline': 'Fixed-seed Random', **{k: v for k, v in random_run.metrics.items() if k != 'daily_returns'}},
])
display(comparison)
display(buy_hold.history)

,Baseline,initial_portfolio_value,final_portfolio_value,total_return,maximum_drawdown,number_of_trades,total_transaction_costs,sharpe_ratio,annualized_volatility
0,Buy and Hold,1000000.0,1.125028e+06,0.125028,0.004617,1,1498.235062,14.306056,0.026312
1,Fixed-seed Random,1000000.0,1.032178e+06,0.032178,0.011117,29,44047.377302,3.232349,0.031414


,initial_portfolio_value,observation_date,execution_date,action,action_name,execution_price,shares_traded,transaction_cost,cash,shares_held,portfolio_value,realized_profit_loss,unrealized_profit_loss,drawdown,reward
0,1000000.0,2025-01-01,2025-01-02,1,Buy,100.149352,9975,1498.235062,11.220571,9975,1.001494e+06,0.0,1494.264938,0.0,0.001493
1,1000000.0,2025-01-02,2025-01-03,0,Hold,NaN,0,0.000000,11.220571,9975,1.005324e+06,0.0,5323.870066,0.0,0.003817
2,1000000.0,2025-01-03,2025-01-06,0,Hold,NaN,0,0.000000,11.220571,9975,1.008856e+06,0.0,8856.136823,0.0,0.003507
3,1000000.0,2025-01-06,2025-01-07,0,Hold,NaN,0,0.000000,11.220571,9975,1.011966e+06,0.0,11965.653138,0.0,0.003077
4,1000000.0,2025-01-07,2025-01-08,0,Hold,NaN,0,0.000000,11.220571,9975,1.014553e+06,0.0,14553.291516,0.0,0.002554
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,1000000.0,2025-04-15,2025-04-16,0,Hold,NaN,0,0.000000,11.220571,9975,1.110159e+06,0.0,110159.030397,0.0,0.003555
75,1000000.0,2025-04-16,2025-04-17,0,Hold,NaN,0,0.000000,11.220571,9975,1.114161e+06,0.0,114160.676117,0.0,0.003598
76,1000000.0,2025-04-17,2025-04-18,0,Hold,NaN,0,0.000000,11.220571,9975,1.118069e+06,0.0,118069.368414,0.0,0.003502
77,1000000.0,2025-04-18,2025-04-21,0,Hold,NaN,0,0.000000,11.220571,9975,1.121736e+06,0.0,121736.290898,0.0,0.003274


## Portfolio-value plot

In [7]:
plot_data = pd.concat([buy_hold.history.assign(Baseline='Buy and Hold'), random_run.history.assign(Baseline='Fixed-seed Random')])
alt.Chart(plot_data).mark_line().encode(x=alt.X('execution_date:T', title='Trading Date'), y=alt.Y('portfolio_value:Q', title='Portfolio Value (PKR)'), color='Baseline:N', tooltip=['Baseline', 'execution_date:T', 'portfolio_value:Q']).properties(width=750, height=350)

alt.Chart(...)

## Leakage and accounting sanity checks

In [8]:
validation = validate_environment(SingleSymbolTradingEnv(dataset, config))
assert validation.valid, validation.errors
history = buy_hold.history
assert (history['observation_date'] < history['execution_date']).all()
assert np.allclose(history['portfolio_value'], history['cash'] + history['shares_held'] * dataset.set_index(pd.to_datetime(dataset['date'])).loc[pd.to_datetime(history['execution_date']), 'close'].to_numpy())
assert (history['cash'] >= 0).all() and (history['shares_held'] >= 0).all()
display(validation)

EnvironmentValidationResult(environment_version='single_symbol_env_v1', valid=True, observation_shape=(17,), errors=())

## Limitations and PPO readiness
Environment v1 is a deterministic single-symbol, long-only, all-in/all-out simulator. It excludes short selling, leverage, fractional shares, exact broker fee schedules, liquidity/market-impact modelling, corporate actions, and multi-asset allocation. Baselines are not AI models. PPO training, tuning, walk-forward evaluation, and registry writes are deferred to Milestone 5B. Demonstration-fixture results are mechanical checks only.